# 🎛️ TIL Metric Scenario Lab

### Mexa no cenário. Observe as métricas. Explique o que aconteceu.

Este laboratório complementa a **Aula 13 — Métricas e Indicadores**.

A regra mais importante é: **você não move Accuracy, Precision, Recall ou F1 diretamente**. Você altera as condições do problema — como `threshold`, prevalência e custos — e as métricas são recalculadas a partir das decisões simuladas.


## 📘 Glossário Vivo — sua bússola durante o laboratório

**[Matriz de confusão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#matriz-de-confusão) · [Acurácia](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#acurácia) · [Precisão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#precisão) · [Recall](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#recall) · [F1-score](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/f1-and-harmonic-mean.pt-BR.md#f1-score) · [Média harmônica](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/f1-and-harmonic-mean.pt-BR.md#média-harmônica) · [Falso positivo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falso-positivo) · [Falso negativo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falso-negativo) · [Especificidade](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#especificidade) · [Balanced Accuracy](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#balanced-accuracy) · [Support](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#support) · [Threshold](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#threshold-de-decisão)**

### 🧭 Se você observar... consulte...

| Sintoma no laboratório | Conceitos do Glossário |
|---|---|
| muitos positivos escapando | Recall · Falso negativo |
| muitos alertas incorretos | Precisão · Falso positivo |
| acurácia alta com classe rara | Acurácia · Support · Balanced Accuracy |
| mudança forte ao mover o corte | Threshold |
| F1 menor do que você esperava | F1-score · Média harmônica |

> Use o Glossário quando o painel produzir um comportamento que surpreenda você.


## 💰 Desenvolvendo sensibilidade para custo

Em Machine Learning, **erro não é apenas um número na matriz de confusão**. Ele pode representar dinheiro, tempo, risco, sofrimento ou até perda de vida.

| Situação | Erro possível | Consequência prática |
|---|---|---|
| triagem de doença grave | falso negativo | uma pessoa doente pode não ser encaminhada; em casos extremos, o custo humano pode ser a perda de uma vida |
| fraude bancária | falso negativo | fraude não bloqueada e perda financeira |
| fraude bancária | falso positivo | compra legítima bloqueada, constrangimento e possível abandono do serviço |
| filtro de spam | falso positivo | e-mail legítimo importante vai para spam |
| reclamação crítica | falso negativo | caso urgente não recebe prioridade |
| moderação | falso positivo | conteúdo legítimo é removido ou usuário é penalizado injustamente |
| automação documental | classificação errada | retrabalho, atraso ou encaminhamento incorreto |

### Um cuidado essencial

Nem todo custo deve ser convertido de maneira simplista para dinheiro. Em saúde e segurança, **uma vida não é apenas um valor monetário em uma planilha**. Há dimensões éticas, regulatórias, sociais e humanas que precisam permanecer explícitas.

Os valores monetários deste laboratório são **proxies didáticos** para tornar trade-offs observáveis. Eles não representam o valor real de uma vida, de um dano grave ou de uma consequência social.

> **Pergunta-chave:** se eu errar, o que acontece no mundo real — e com quem?


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

SEED = 42
N_CASES = 2000
rng = np.random.default_rng(SEED)
BASE_POS = rng.normal(0, 1, N_CASES)
BASE_NEG = rng.normal(0, 1, N_CASES)


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def safe_div(num, den):
    return num / den if den else 0.0

def simulate(prevalence, separation, threshold):
    n_pos = max(1, int(round(N_CASES * prevalence)))
    n_neg = N_CASES - n_pos
    pos_scores = sigmoid(BASE_POS[:n_pos] + separation)
    neg_scores = sigmoid(BASE_NEG[:n_neg] - separation)
    y_true = np.concatenate([np.ones(n_pos, dtype=int), np.zeros(n_neg, dtype=int)])
    scores = np.concatenate([pos_scores, neg_scores])
    y_pred = (scores >= threshold).astype(int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    return {'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn}

def metrics_from_cm(cm):
    tp, fp, fn, tn = cm['TP'], cm['FP'], cm['FN'], cm['TN']
    accuracy = safe_div(tp + tn, tp + fp + fn + tn)
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    f1 = safe_div(2 * precision * recall, precision + recall)
    balanced_accuracy = (recall + specificity) / 2
    return {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1': f1, 'Specificity': specificity, 'Balanced Accuracy': balanced_accuracy}

def expected_cost(cm, cost_fp, cost_fn, review_cost, inference_cost):
    predicted_positive = cm['TP'] + cm['FP']
    return cm['FP'] * cost_fp + cm['FN'] * cost_fn + predicted_positive * review_cost + N_CASES * inference_cost

def metric_bar(label, value):
    pct = max(0, min(100, value * 100))
    return f'''<div style="margin:8px 0"><div style="display:flex;justify-content:space-between;font-weight:600"><span>{label}</span><span>{value:.3f}</span></div><div style="height:13px;background:#e9ecef;border-radius:8px;overflow:hidden"><div style="width:{pct:.1f}%;height:100%;background:#4a77b5"></div></div></div>'''

def interpret(metrics, cm, prevalence, threshold, cost_fp, cost_fn, total_cost):
    notes = []
    if prevalence < 0.10 and metrics['Accuracy'] > 0.90:
        notes.append('Classe positiva rara + acurácia alta: não interprete Accuracy isoladamente.')
    if metrics['Recall'] < 0.60:
        notes.append('Recall baixo: muitos positivos reais estão escapando.')
    elif metrics['Recall'] >= 0.85:
        notes.append('Recall alto: poucos positivos reais estão escapando.')
    if metrics['Precision'] < 0.60:
        notes.append('Precision baixa: muitos alertas positivos são falsos.')
    elif metrics['Precision'] >= 0.85:
        notes.append('Precision alta: os alertas positivos tendem a ser confiáveis.')
    if cost_fn >= 5 * max(cost_fp, 0.01):
        notes.append('FN é muito mais caro que FP; baixar o threshold pode ser justificável para ganhar Recall.')
    elif cost_fp >= 5 * max(cost_fn, 0.01):
        notes.append('FP é muito mais caro; subir o threshold pode ser justificável para reduzir falsos alertas.')
    notes.append(f"FP={cm['FP']}, FN={cm['FN']}, F1={metrics['F1']:.3f}, custo estimado=R$ {total_cost:,.2f}.")
    notes.append("📘 Para entender por que o F1 cai quando Precision ou Recall ficam desequilibrados, consulte F1-score e Média harmônica no Glossário.")
    return '<br><br>'.join('• ' + n for n in notes)


In [ ]:
SCENARIOS = {
    'Fraude bancária': dict(prevalence=.05, separation=1.15, threshold=.50, cost_fp=10., cost_fn=250., review_cost=4., inference_cost=.02),
    'Triagem médica': dict(prevalence=.12, separation=1.00, threshold=.40, cost_fp=25., cost_fn=500., review_cost=18., inference_cost=.03),
    'Filtro de spam': dict(prevalence=.30, separation=1.25, threshold=.60, cost_fp=60., cost_fn=3., review_cost=0., inference_cost=.005),
    'Reclamação crítica': dict(prevalence=.08, separation=.95, threshold=.45, cost_fp=8., cost_fn=120., review_cost=7., inference_cost=.01),
}

class MetricScenarioLab:
    def __init__(self):
        self.scenario = widgets.Dropdown(options=list(SCENARIOS), value='Fraude bancária', description='Cenário:', layout=widgets.Layout(width='420px'))
        self.threshold = widgets.FloatSlider(min=.05, max=.95, step=.05, value=.50, description='Threshold:', readout_format='.2f', continuous_update=False, layout=widgets.Layout(width='520px'))
        self.prevalence = widgets.FloatSlider(min=.01, max=.50, step=.01, value=.05, description='Prevalência:', readout_format='.0%', continuous_update=False, layout=widgets.Layout(width='520px'))
        self.separation = widgets.FloatSlider(min=.30, max=2.50, step=.05, value=1.15, description='Separação:', readout_format='.2f', continuous_update=False, layout=widgets.Layout(width='520px'))
        self.cost_fp = widgets.FloatSlider(min=0, max=300, step=5, value=10, description='Custo FP:', continuous_update=False, layout=widgets.Layout(width='520px'))
        self.cost_fn = widgets.FloatSlider(min=0, max=800, step=10, value=250, description='Custo FN:', continuous_update=False, layout=widgets.Layout(width='520px'))
        self.review_cost = widgets.FloatSlider(min=0, max=100, step=1, value=4, description='Revisão:', continuous_update=False, layout=widgets.Layout(width='520px'))
        self.inference_cost = widgets.FloatSlider(min=0, max=1, step=.005, value=.02, description='Inferência:', readout_format='.3f', continuous_update=False, layout=widgets.Layout(width='520px'))
        self.metrics_html = widgets.HTML(); self.cost_html = widgets.HTML(); self.interpretation_html = widgets.HTML()
        self.matrix_output = widgets.Output(); self.snapshot_output = widgets.Output()
        self.best_f1_button = widgets.Button(description='🎯 Melhor F1')
        self.min_cost_button = widgets.Button(description='💰 Menor custo')
        self.snapshot_button = widgets.Button(description='📸 Snapshot estático')
        self.reset_button = widgets.Button(description='↺ Restaurar')
        self.scenario.observe(self._load_scenario, names='value')
        for c in self._controls(): c.observe(self._update, names='value')
        self.best_f1_button.on_click(lambda _: self._search_threshold('f1'))
        self.min_cost_button.on_click(lambda _: self._search_threshold('cost'))
        self.snapshot_button.on_click(self._snapshot)
        self.reset_button.on_click(lambda _: self._load_scenario({'new': self.scenario.value}))
        self._load_scenario({'new': self.scenario.value})

    def _controls(self):
        return [self.threshold, self.prevalence, self.separation, self.cost_fp, self.cost_fn, self.review_cost, self.inference_cost]

    def _load_scenario(self, change):
        cfg = SCENARIOS[change['new']]
        for c in self._controls(): c.unobserve(self._update, names='value')
        self.prevalence.value=cfg['prevalence']; self.separation.value=cfg['separation']; self.threshold.value=cfg['threshold']
        self.cost_fp.value=cfg['cost_fp']; self.cost_fn.value=cfg['cost_fn']; self.review_cost.value=cfg['review_cost']; self.inference_cost.value=cfg['inference_cost']
        for c in self._controls(): c.observe(self._update, names='value')
        self._update()

    def _state(self, threshold=None):
        t = self.threshold.value if threshold is None else threshold
        cm = simulate(self.prevalence.value, self.separation.value, t)
        metrics = metrics_from_cm(cm)
        cost = expected_cost(cm, self.cost_fp.value, self.cost_fn.value, self.review_cost.value, self.inference_cost.value)
        return cm, metrics, cost

    def _update(self, change=None):
        cm, metrics, cost = self._state()
        bars = ''.join(metric_bar(k, v) for k, v in metrics.items())
        self.metrics_html.value = f'<div style="padding:14px;border:1px solid #ddd;border-radius:12px"><h4>Indicadores</h4>{bars}</div>'
        self.cost_html.value = f"<div style='padding:14px;border:1px solid #ddd;border-radius:12px;margin-top:10px'><b>Erros:</b> FP={cm['FP']} · FN={cm['FN']}<br><b>Custo esperado:</b> R$ {cost:,.2f}<br><b>Custo médio/decisão:</b> R$ {cost/N_CASES:,.4f}<br><br><small>⚠️ Custos são proxies didáticos. Em saúde, segurança e direitos, não reduza consequências humanas a um único valor monetário.</small></div>"
        explanation = interpret(metrics, cm, self.prevalence.value, self.threshold.value, self.cost_fp.value, self.cost_fn.value, cost)
        self.interpretation_html.value = f'<div style="padding:14px;border:1px solid #ddd;border-radius:12px"><h4>Leitura do cenário</h4>{explanation}</div>'
        with self.matrix_output:
            clear_output(wait=True)
            matrix = np.array([[cm['TN'], cm['FP']], [cm['FN'], cm['TP']]])
            fig, ax = plt.subplots(figsize=(5.2, 4.0)); ax.imshow(matrix)
            ax.set_xticks([0,1], labels=['Negativo','Positivo']); ax.set_yticks([0,1], labels=['Negativo','Positivo'])
            ax.set_xlabel('Predito'); ax.set_ylabel('Real'); ax.set_title('Matriz de confusão')
            for i in range(2):
                for j in range(2): ax.text(j, i, matrix[i,j], ha='center', va='center')
            plt.show(); plt.close(fig)

    def _search_threshold(self, objective):
        rows=[]
        for t in np.linspace(.05,.95,91):
            cm, metrics, cost = self._state(float(t)); rows.append((float(t), metrics['F1'], cost))
        chosen = max(rows, key=lambda x:x[1]) if objective=='f1' else min(rows, key=lambda x:x[2])
        self.threshold.value = round(chosen[0],2)

    def _snapshot(self, _):
        cm, metrics, cost = self._state()
        with self.snapshot_output:
            clear_output(wait=True)
            display(pd.DataFrame({'Indicador':list(metrics)+['FP','FN','Custo esperado'],'Valor':list(metrics.values())+[cm['FP'],cm['FN'],cost]}))

    def show(self):
        controls = widgets.VBox([self.scenario, widgets.HTML('<b>Comportamento do classificador</b>'), self.threshold, self.prevalence, self.separation, widgets.HTML('<b>Custos do cenário</b>'), self.cost_fp, self.cost_fn, self.review_cost, self.inference_cost, widgets.HBox([self.best_f1_button,self.min_cost_button,self.reset_button]), self.snapshot_button])
        dashboard = widgets.HBox([widgets.VBox([self.metrics_html,self.cost_html], layout=widgets.Layout(width='48%')), widgets.VBox([self.interpretation_html,self.matrix_output], layout=widgets.Layout(width='50%'))], layout=widgets.Layout(width='100%', justify_content='space-between', align_items='flex-start'))
        display(controls); display(dashboard); display(self.snapshot_output)

lab = MetricScenarioLab()
lab.show()


### Evidência estática antes da interação

O painel interativo é uma camada de exploração. Para manter o notebook **headless-first**, o cenário padrão abaixo também é calculado de forma determinística e deve aparecer em uma execução `Run All` sem qualquer clique.


In [ ]:
cfg = SCENARIOS['Fraude bancária']
cm_default = simulate(cfg['prevalence'], cfg['separation'], cfg['threshold'])
metrics_default = metrics_from_cm(cm_default)
cost_default = expected_cost(
    cm_default,
    cfg['cost_fp'],
    cfg['cost_fn'],
    cfg['review_cost'],
    cfg['inference_cost']
)

static_summary = pd.DataFrame({
    'Indicador': list(metrics_default) + ['FP', 'FN', 'Custo esperado'],
    'Valor': list(metrics_default.values()) + [cm_default['FP'], cm_default['FN'], cost_default]
})

display(static_summary)


## 🧠 Exercício de sensibilidade ao custo

**1. Triagem médica** — Se um falso negativo significar que uma pessoa com doença grave não é encaminhada, você aceitaria aumentar falsos positivos para reduzir falsos negativos?

**2. Fraude bancária** — O que é pior: bloquear 100 compras legítimas ou deixar passar 5 fraudes? A resposta depende apenas da soma em reais?

**3. Reclamação crítica** — Se um falso negativo atrasar o atendimento de uma situação urgente, como você representaria esse custo além de dinheiro?

**4. Spam** — Por que o custo relativo de FP e FN pode ser quase invertido em comparação com uma triagem médica?

> O objetivo é desenvolver **sensibilidade ao impacto**, não decorar que FN ou FP é sempre pior.


## Missões de exploração

**1 — Threshold:** mova o threshold e tente prever o sentido de Recall, Precision, FP, FN e custo.

**2 — Contexto:** compare Triagem médica com Filtro de spam. Qual erro é menos tolerável em cada caso?

**3 — Melhor F1 ≠ menor custo:** use **🎯 Melhor F1** e **💰 Menor custo**. Depois consulte [F1-score](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/f1-and-harmonic-mean.pt-BR.md#f1-score) e [Média harmônica](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/f1-and-harmonic-mean.pt-BR.md#média-harmônica).

**4 — Prevalência:** reduza a prevalência e observe por que Accuracy pode ficar alta e ainda assim ser pouco informativa.


## Limites do simulador

Este é um **simulador didático**, não um estimador de desempenho real. Os scores são sintéticos e os custos são exemplos. Thresholds reais exigem dados de validação representativos.

> **O painel existe para treinar julgamento — e o Glossário para transformar surpresa em compreensão.**


## Próximo passo — da métrica ao sistema

Depois de explorar thresholds e custos de erro, avance para a **Aula 13C — Model Routing, Orchestration e Utility**.

A pergunta muda de:

> Qual threshold produz a melhor combinação de métricas?

para:

> Qual arquitetura entrega qualidade suficiente com custo, latência e risco aceitáveis?

Na 13C, a interatividade segue o padrão **headless-first**: `Run All` deve terminar sem intervenção humana e os controles interativos são uma camada opcional de exploração.
